# GitHub Repo Explorer

Interactive parallel coordinates plot for selecting evaluation repos.

## Setup

1. Run the fetch script first:
   ```bash
   export GITHUB_TOKEN=ghp_...
   uv run python evals/eda/fetch_repos.py --csv
   ```
2. Then open this notebook.

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import clear_output, display

In [ ]:
# Load data — try parquet first, fall back to CSV
data_dir = (
    Path()
    if Path("repos.parquet").exists()
    else Path(__file__).parent
    if "__file__" in dir()
    else Path()
)
parquet = data_dir / "repos.parquet"
csv = data_dir / "repos.csv"

if parquet.exists():
    df = pd.read_parquet(parquet)
elif csv.exists():
    df = pd.read_csv(csv)
else:
    raise FileNotFoundError("Run fetch_repos.py first to generate repos.parquet")

print(f"Loaded {len(df)} repos")
df.head()

In [ ]:
# Assign a numeric language code for coloring
languages = sorted(df["language"].dropna().unique())
lang_map = {lang: i for i, lang in enumerate(languages)}
df["lang_code"] = df["language"].map(lang_map).fillna(-1).astype(int)

# Log-scale helpers for skewed columns
for col in ["stars", "forks", "size_mb", "total_commits", "recent_commits_90d", "open_issues", "open_prs"]:
    df[f"log_{col}"] = np.log1p(df[col])

In [ ]:
import json as _json

# ── Parallel coordinates plot with ipywidgets ──

# Axes to show (log-scaled versions of the skewed metrics)
axes = [
    {
        "col": "lang_code",
        "label": "Language",
        "tickvals": list(lang_map.values()),
        "ticktext": list(lang_map.keys()),
    },
    {"col": "log_stars", "label": "Stars (log)"},
    {"col": "log_forks", "label": "Forks (log)"},
    {"col": "log_size_mb", "label": "Size MB (log)"},
    {"col": "log_total_commits", "label": "Total Commits (log)"},
    {"col": "log_recent_commits_90d", "label": "Recent Commits 90d (log)"},
    {"col": "log_open_issues", "label": "Open Issues (log)"},
    {"col": "log_open_prs", "label": "Open PRs (log)"},
]

dimensions = []
for ax in axes:
    d = {"label": ax["label"], "values": df[ax["col"]]}
    if "tickvals" in ax:
        d["tickvals"] = ax["tickvals"]
        d["ticktext"] = ax["ticktext"]
    dimensions.append(d)

fig = go.FigureWidget(
    data=go.Parcoords(
        line={
            "color": df["lang_code"],
            "colorscale": "Turbo",
            "showscale": True,
            "cmin": df["lang_code"].min(),
            "cmax": df["lang_code"].max(),
        },
        dimensions=dimensions,
        unselected={"line": {"color": "lightgray", "opacity": 0.3}},
    ),
    layout={
        "title": "GitHub Repos - Parallel Coordinates",
        "width": 1200,
        "height": 600,
        "margin": {"l": 80, "r": 80, "t": 60, "b": 30},
    },
)

# Output area for selected repos
selection_output = widgets.Output(layout=widgets.Layout(max_height="400px", overflow_y="auto"))

export_btn = widgets.Button(
    description="Export selected → JSONL", button_style="success", icon="download"
)
status_label = widgets.Label(value="Drag ranges on axes above to filter repos")


def _get_selected_mask() -> pd.Series:
    """Determine which rows pass the current constraint ranges."""
    mask = pd.Series(True, index=df.index)
    parcoords = fig.data[0]
    for i, dim in enumerate(parcoords.dimensions):
        cr = dim.constraintrange
        if cr is not None:
            col = axes[i]["col"]
            # constraintrange can be [lo, hi] or [[lo,hi],[lo,hi],...]
            if isinstance(cr[0], list | tuple):
                sub = pd.Series(False, index=df.index)
                for lo, hi in cr:
                    sub |= (df[col] >= lo) & (df[col] <= hi)
                mask &= sub
            else:
                lo, hi = cr
                mask &= (df[col] >= lo) & (df[col] <= hi)
    return mask


def _update_selection(*_args):
    mask = _get_selected_mask()
    selected = df.loc[
        mask,
        [
            "repo",
            "language",
            "stars",
            "size_mb",
            "recent_commits_90d",
            "total_commits",
            "forks",
            "open_issues",
            "url",
        ],
    ]
    with selection_output:
        clear_output(wait=True)
        status_label.value = f"{len(selected)} repos selected"
        if len(selected) > 0:
            display(selected.sort_values("stars", ascending=False).reset_index(drop=True))


# Watch for constraint range changes on each dimension
for dim in fig.data[0].dimensions:
    dim.on_change(_update_selection, "constraintrange")


def _export_clicked(_btn):
    mask = _get_selected_mask()
    selected = df.loc[mask].copy()
    # Build JSONL matching the format of evals/examples/repos.jsonl
    out_path = Path("selected_repos.jsonl")
    with out_path.open("w") as f:
        for rank, (_, row) in enumerate(
            selected.sort_values("stars", ascending=False).iterrows(), 1
        ):
            rec = {
                "rank": rank,
                "repo": row["url"],
                "language": row.get("language", ""),
                "stars": int(row["stars"]),
                "size_mb": float(row["size_mb"]),
                "recent_commits_90d": int(row.get("recent_commits_90d", 0)),
                "notes": row.get("description", ""),
            }
            f.write(_json.dumps(rec) + "\n")
    status_label.value = f"Exported {len(selected)} repos → {out_path}"


export_btn.on_click(_export_clicked)

display(
    widgets.VBox(
        [
            fig,
            widgets.HBox([status_label, export_btn]),
            selection_output,
        ]
    )
)

# Initial population
_update_selection()